## Forecasting Appointment Volume for Nexora Care Flow

## Company Overview
Nexora Care Flow is an all-in-one software tool made by Nexora Care.

## The Business Problem
1. Small clinics face unpredictable changes in patient visits and have no way to see what is coming:
2. Clinic managers schedule staff and rooms based on gut feeling or simple past averages.
3. Clinics are understaffed when there is surge, leading to stressed workers, long patient wait times, and delayed care.
4. Clinics are overstaffed during slow days, wasting labor and money.
The system already collects appointment data, but it is not being used to predict the future.

## Project Purpose
Turn existing clinic data into a demand forecasting tool. By tracking day-of-week trends, holidays, and illness seasons (like flu season), the tool helps clinics plan staff and room needs before the week starts.


## Project Objectives
1. Analyze past appointment data to identify recurring patterns, such as days of the week, holidays, and illness seasons; that drive patient volume shifts.
2. Build a forward-looking time-series model that uses historical clinic data to predict appointment volumes several weeks in advance.
3. Turn the forecast results into simple, practical staffing guidance and warnings so managers can schedule staff and rooms proactively.
4. Set up a documented process to regularly compare predictions with actual visits and retrain the model as new data arrives.


## Expected Results
1. Clinic managers stop guessing and start planning ahead.
2. Less money wasted on idle staff or extra overtime pay.
3. Shorter wait times and less stress for clinic workers.

In [43]:
# import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [44]:
# load the dataset
df=pd.read_csv(r"C:\Users\Lenovo 2022\OneDrive\Documents\Nexora_care_flow\data\raw\AppointmentRecords.csv")

In [45]:
# row and column counts
print(df.shape)

(129353, 15)


In [46]:
# data types and null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129353 entries, 0 to 129352
Data columns (total 15 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   AppointmentID         129353 non-null  int64 
 1   ClinicID              129353 non-null  int64 
 2   ClinicName            129353 non-null  object
 3   ClinicType            129353 non-null  object
 4   PatientID             129353 non-null  int64 
 5   ProviderID            129353 non-null  int64 
 6   AppointmentDateTime   129353 non-null  object
 7   AppointmentType       129353 non-null  object
 8   Status                129353 non-null  object
 9   BookingChannel        129353 non-null  object
 10  BookingLeadTimeDays   129353 non-null  int64 
 11  IsHoliday             129353 non-null  bool  
 12  HolidayName           939 non-null     object
 13  SeasonFlag            129353 non-null  object
 14  ChronicConditionFlag  129353 non-null  bool  
dtypes: bool(2), int64

In [47]:
df.head()

,AppointmentID,ClinicID,ClinicName,ClinicType,PatientID,ProviderID,AppointmentDateTime,AppointmentType,Status,BookingChannel,BookingLeadTimeDays,IsHoliday,HolidayName,SeasonFlag,ChronicConditionFlag
0,529385,2,Nexora Care - Lakeside,Multi-Specialty,103642,202,2024-01-01 08:00,New Patient,Completed,Online,2,True,New Year's Day,Flu Season,False
1,500000,1,Nexora Care - Riverside,Primary Care,101551,101,2024-01-01 08:30,New Patient,No-Show,Online,18,True,New Year's Day,Flu Season,False
2,500011,1,Nexora Care - Riverside,Primary Care,100260,104,2024-01-01 08:30,Follow-Up,Completed,Referral,4,True,New Year's Day,Flu Season,False
3,575066,3,Nexora Care - Downtown Express,Telehealth-Forward,104923,301,2024-01-01 09:45,Urgent,Completed,Online,0,True,New Year's Day,Flu Season,False
4,500008,1,Nexora Care - Riverside,Primary Care,101124,104,2024-01-01 09:45,Telehealth,Completed,Phone,2,True,New Year's Day,Flu Season,False


In [28]:
df.describe()

,AppointmentID,ClinicID,PatientID,ProviderID,BookingLeadTimeDays
count,129353.000000,129353.000000,129353.000000,129353.000000,129353.000000
mean,564676.000000,2.417864,103553.485385,245.290005,8.704267
std,37341.139023,1.071525,2020.941152,107.165883,7.641921
min,500000.000000,1.000000,100000.000000,101.000000,0.000000
25%,532338.000000,2.000000,101812.000000,201.000000,3.000000
50%,564676.000000,2.000000,103547.000000,205.000000,7.000000
75%,597014.000000,3.000000,105178.000000,306.000000,12.000000
max,629352.000000,4.000000,107279.000000,406.000000,60.000000


In [48]:
df.nunique()

AppointmentID           129353
ClinicID                     4
ClinicName                   4
ClinicType                   3
PatientID                 7109
ProviderID                  24
AppointmentDateTime      30287
AppointmentType              4
Status                       4
BookingChannel               4
BookingLeadTimeDays         61
IsHoliday                    2
HolidayName                 13
SeasonFlag                   3
ChronicConditionFlag         2
dtype: int64

In [49]:
df.isnull().sum()

AppointmentID                0
ClinicID                     0
ClinicName                   0
ClinicType                   0
PatientID                    0
ProviderID                   0
AppointmentDateTime          0
AppointmentType              0
Status                       0
BookingChannel               0
BookingLeadTimeDays          0
IsHoliday                    0
HolidayName             128414
SeasonFlag                   0
ChronicConditionFlag         0
dtype: int64

In [50]:
# Filter rows where IsHoliday is True but HolidayName is missing (NaN)
mismatch = df[(df["IsHoliday"] == True) & (df["HolidayName"].isnull())]

# View the matching rows
mismatch.head()

,AppointmentID,ClinicID,ClinicName,ClinicType,PatientID,ProviderID,AppointmentDateTime,AppointmentType,Status,BookingChannel,BookingLeadTimeDays,IsHoliday,HolidayName,SeasonFlag,ChronicConditionFlag


In [51]:
# Replace null values with 'None'
df["HolidayName"] = df["HolidayName"].fillna("None")

In [52]:
# verify that there are no more null values in the HolidayName column
df["HolidayName"].isnull().sum()

np.int64(0)

In [54]:
# Convert to datetime
df["AppointmentDateTime"] = pd.to_datetime(df["AppointmentDateTime"])

In [55]:
# verifydata types and null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129353 entries, 0 to 129352
Data columns (total 15 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   AppointmentID         129353 non-null  int64         
 1   ClinicID              129353 non-null  int64         
 2   ClinicName            129353 non-null  object        
 3   ClinicType            129353 non-null  object        
 4   PatientID             129353 non-null  int64         
 5   ProviderID            129353 non-null  int64         
 6   AppointmentDateTime   129353 non-null  datetime64[ns]
 7   AppointmentType       129353 non-null  object        
 8   Status                129353 non-null  object        
 9   BookingChannel        129353 non-null  object        
 10  BookingLeadTimeDays   129353 non-null  int64         
 11  IsHoliday             129353 non-null  bool          
 12  HolidayName           129353 non-null  object        
 13 

In [58]:
# Save into the 'data' folder
df.to_csv("C:\\Users\\Lenovo 2022\\OneDrive\\Documents\\Nexora_care_flow\\data\\processed\\cleaned_AppointmentRecords.csv", index=False)